# Лабораторна робота №5 — моніторинг моделі машинного навчання

**Варіант 2.** Impact of AI on Students, прогноз ризику вигорання `Low/Medium/High`.

**Мета (методичка, с. 87):** дослідити data drift/target drift і продуктивність, використовуючи Evidently AI та записи навчання й API із SQLite. Жодні тестові дані не генеруються.

## 1. Підготовка даних

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from app.services.monitoring import prepare, run_evidently
reference, current, audit = prepare(output_dir=ROOT / "reports")
print("Еталон:", len(reference), "записів")
print("Поточні запити:", len(current), "; унікальних:", audit["unique_current_feature_vectors"])
print("Відомі справжні мітки:", audit["known_current_target_count"])
print("12 спільних ознак:", reference.columns.intersection(current.columns).tolist())

## 2. Побудова дашбордів Evidently AI
Після встановлення пакета `evidently` запускаємо звіти `DataDriftPreset` та `DataSummaryPreset`. Вони створюють два HTML-файли саме бібліотекою Evidently, без іншого інструменту.

In [ ]:
files = run_evidently(reference, current, ROOT / "reports")
for path in files:
    print("Звіт Evidently:", path)

## 3. Аналіз результатів

За початковою БД ЛР4: **45 000** навчальних записів і **4** збережені API-запити, серед яких лише **один унікальний вектор ознак**. У двох запитах значення `actual` є відомим, у двох — `NULL`.

- **Data drift:** статистично надійно визначити неможливо через малу й повторювану поточну вибірку; вивід Evidently слід тлумачити лише як діагностичний.
- **Найбільші відхилення:** за потреби їх можна описати через різниці середніх у `audit.json`, але не називати підтвердженим дрейфом.
- **Target drift:** два відомі значення не дають підстав стверджувати, що розподіл класів змінився.
- **Деградація моделі:** за 2 мітками в різних навчальних запусках не доведена і не спростована.
- **Необхідність перенавчання:** підтверджувальних даних недостатньо.

**Висновок.** Підготовлено SQL-вибірки та процедуру побудови Evidently-звітів для реальних даних із попередньої роботи. Критичне обмеження — недостатня кількість нових унікальних запитів та фактичних міток, тому достовірні висновки щодо дрейфу і деградації поки неможливі. Для завершення повного аналізу потрібно зібрати додаткові незалежні запити API; жодні результати не моделювалися штучно.